In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join(r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src'))

if module_path not in sys.path:
    sys.path.append(module_path)

In [31]:
import numpy as np
from typing import Any, Dict

REQUEST_BYTE_LIMIT = 2**20 * 48
print(REQUEST_BYTE_LIMIT)
import numpy as np

def auto_chunks(
    dtype_bytes: int, index: int, request_byte_limit: int
) -> Dict[str, int]:
    """Given the data type size, a fixed index size, and request limit, calculate optimal chunks."""
    # Calculate the byte size used by the given index
    index_byte_size = index * dtype_bytes
    
    # Check if the index size alone exceeds the request_byte_limit
    if index_byte_size >= request_byte_limit:
        raise ValueError("The given index size exceeds or nearly exhausts the request byte limit.")

    # Calculate the remaining bytes available for width and height dimensions
    remaining_bytes = request_byte_limit - index_byte_size
    
    # Logarithmic splitting of remaining bytes into width and height, adjusted for dtype size
    log_remaining = np.log2(remaining_bytes / dtype_bytes)  # Directly account for dtype_bytes

    # Divide log_remaining between width and height
    d = log_remaining / 2
    wd, ht = np.ceil(d), np.floor(d)

    # Convert width and height from log space to actual values
    width = int(2 ** wd)
    height = int(2 ** ht)

    # Recheck if the final size exceeds the request_byte_limit and adjust
    total_bytes = index * width * height * dtype_bytes
    while total_bytes > request_byte_limit:
        # If the total size exceeds, scale down width and height by reducing one of them
        if width > height:
            width //= 2
        else:
            height //= 2
        total_bytes = index * width * height * dtype_bytes

    actual_bytes = index * width * height * dtype_bytes
    if actual_bytes > request_byte_limit:
        raise ValueError(
            f'`chunks="auto"` failed! Actual bytes {actual_bytes!r} exceeds limit'
            f' {request_byte_limit!r}.  Please choose another value for `chunks` (and file a'
            ' bug).'
        )
    
    return {'index': index, 'width': width, 'height': height}


eight_byte_chunk = auto_chunks(dtype_bytes=8, index=48, request_byte_limit=REQUEST_BYTE_LIMIT)
print(eight_byte_chunk)

50331648
{'index': 49, 'width': 256, 'height': 256}


In [2]:
ee_key_host_path =r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json' # Path in the host machine
ee_key_container_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

volumes = {
    ee_key_host_path: {'bind': ee_key_container_path, 'mode': 'ro'},  # 'ro' means read-only
}
from dask_cluster_manager import DaskClusterManager

dcm = DaskClusterManager()
dcm.create_test_cluster(volumes=volumes)

Dask Scheduler started with ID 3f77b92f17fdf794d37f6561235829c18380b3694852985f08de8725f8106bd9
Dask Worker started with ID 380a4d865284020e8b567169df61544b5e7e96fb4eeb10f57083c971095478a3
Test cluster created with 1 worker and half the system memory.
Dask dashboard available at http://localhost:8787


In [3]:
from dask.distributed import Client
dask_client = Client("tcp://localhost:8786")

In [15]:
dcm.stop_and_remove_containers()

Dask Worker 574f9b52838be5b366d64d3b73bee4887991431e47f0aa496290940e6fe984dc stopped and removed.
Dask Scheduler 1efc730ce650d6584fb8feeb39f822b7781cc27c53bd6385f9fef73cae9fc1d5 stopped and removed.


In [ ]:
logs = scheduler.logs()
print(logs.decode("utf-8"))

In [4]:
from dask_plugins import EEPlugin

ee_plugin = EEPlugin(ee_key_container_path)
dask_client.register_plugin(ee_plugin)

{'tcp://172.20.0.3:40331': {'status': 'OK'}}

In [5]:
# Earth Engine xarray
import sys
import os
import ee
import json

json_key = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)

ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

if module_path not in sys.path:
    sys.path.append(module_path)

from input_driver import EarthEngineData

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'map_function': prep_sr_l8
        }

earth_engine = EarthEngineData(parameters)
#ds = EarthEngineData(parameters).dataset

In [ ]:
print(ds)

In [7]:
# Template xarray based on Earth Engine
import numpy as np
import pandas as pd
import xarray as xr

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    #'SR_B1': (['time', 'X', 'Y'], data),
    #'SR_B2': (['time', 'X', 'Y'], data),
    #'SR_B3': (['time', 'X', 'Y'], data),
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
    #'SR_B6': (['time', 'X', 'Y'], data),
    #'ST_EMSD': (['time', 'X', 'Y'], data),
    #'ST_QA': (['time', 'X', 'Y'], data),
    #'ST_TRAD': (['time', 'X', 'Y'], data),
    #'ST_URAD': (['time', 'X', 'Y'], data),
    #'QA_PIXEL': (['time', 'X', 'Y'], data),
    #'QA_RADSAT': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 1, 'Y': 1}

# Create the dataset
ds = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001C5C981A170>>
Traceback (most recent call last):
  File "c:\Users\Adriano Matos\anaconda3\envs\robust_raster\lib\site-packages\ipykernel\ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
# Load the data using Dask Delayed!
import xarray as xr
import dask

chunk_size  = {
            'time': 3,
            'X': 512,
            'Y': 256
}
# Define a delayed function that opens the dataset
@dask.delayed
def load_dataset_on_worker():
    return xr.open_dataset('/data/saved_on_disk.nc', chunks=chunk_size)

# Trigger the delayed execution, this will happen on the worker
delayed_ds = load_dataset_on_worker()

# Optionally, you can convert the delayed object into a Dask-backed xarray Dataset
ds = delayed_ds.compute()

# Now ds can be used for further computation on the Dask workers

In [ ]:
import xarray as xr
import dask.array as da
import dask
import psutil
import time

def log_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / 1024**2  # Memory usage in MB


def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

def user_function_wrapper(ds, user_func, *args, **kwargs):
    df_input = ds.to_dataframe().reset_index()
    df_output = user_func(df_input, *args, **kwargs)
    df_output = df_output.set_index(list(ds.dims))
    ds_output = df_output.to_xarray()
    return ds_output


def tune_user_function(chunk, user_func, user_func2, *args, **kwargs):

    result_chunk = user_func2(ds, user_func, *args, kwargs)
    start_time = time.time()
    result_chunk.compute()
    end_time = time.time()
        
    wall_time = end_time - start_time
    memory_usage = log_memory_usage()
    print(f"Wall time: {wall_time:.2f} seconds")
    print(f"Memory usage: {memory_usage:.2f} MB")

    # Apply the processing function to this chunk
    #processed_chunk = user_func(one_chunk)

    return result_chunk

def test_run(ds, user_func, user_func2, *args, **kwargs):
    # Dynamically determine dimension names
    dim_names = list(ds.dims.keys())
            
    # Extract a single chunk to determine the output structure using dynamic dimension names
    one_chunk_slices = {dim: slice(0, ds.chunks[dim][0]) for dim in dim_names}
    one_chunk = ds.isel(**one_chunk_slices)

    test = xr.map_blocks(tune_user_function,
                            one_chunk, 
                            args=(user_func, user_func2) + args, 
                            kwargs=kwargs)

result = test_run(ds, process_chunk_as_pandas, user_function_wrapper)

In [ ]:
import xarray as xr
import pandas as pd
import sys, os

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction()

# Apply the user-defined function
# If I load the dataset as a dask delayed, I don't need a template!
user_defined_func.tune_user_function(earth_engine, process_chunk_as_pandas)

In [ ]:
print(ds_chunked)

In [8]:
import math
import pandas as pd
import psutil

# Read the CSV file into a DataFrame
df = pd.read_csv('metrics_report.csv')

def get_available_system_memory():
    # Get total available RAM in bytes
    total_ram = psutil.virtual_memory().total

    # Convert from bytes to gigabytes (GB)
    total_ram_gb = total_ram / (1024 ** 3)

    return total_ram_gb

# Get the max workers (RAM limited)
available_system_memory = get_available_system_memory()
ram_safety_threshold = 0.5

print(df['wRAM'][0])
print(available_system_memory)
dcm.create_cluster(num_workers=df['wRAM'][0], n_threads=1, memory_limit=f"{available_system_memory}GB")

# I need to have a condition that chunks EE Data if I am doing a test or if I am doing a full run
# If test, then chunk to the size of the dataset
# If not, then chunk to 48 MBs?
# We might be limited to 48MBs in Earth Engine...

1
31.767879486083984


In [ ]:
import xarray as xr
import pandas as pd
import sys, os

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction()

# Apply the user-defined function
# If I load the dataset as a dask delayed, I don't need a template!
result = user_defined_func.apply_user_function(ds, process_chunk_as_pandas)

In [ ]:
print(result)